
# Cantilever Beam CNN Project

This notebook solves both:

- Approach 1 → Regression
- Approach 2 → Classification

using PyTorch CNN.


In [1]:

import os
import json
import joblib
import numpy as np
import pandas as pd

from glob import glob

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader



## Set Paths


In [2]:

dat_dir = "dataset"

xls_pth = os.path.join(dat_dir, "output.xlsx")
txt_dir = os.path.join(dat_dir, "stress")

sav_dir = "runs_cantilever"

os.makedirs(sav_dir, exist_ok=True)



## Load Thickness Data


In [5]:

x_df = pd.read_excel(xls_pth, header=None)

x = x_df.values.astype(np.float32)

print(x.shape)


(5000, 226)



## Load Stress Files


In [6]:

txt_lst = sorted(glob(os.path.join(txt_dir, "*.txt")))

mx_lst = []

for pth in txt_lst:
    
    dat = np.loadtxt(pth)
    
    vm = dat[:, 0]
    
    mx = np.max(vm)
    
    mx_lst.append(mx)

y_reg = np.array(mx_lst, dtype=np.float32)

print(y_reg.shape)


(5000,)



## Make Classification Labels


In [7]:

p10 = np.percentile(y_reg, 10)
p25 = np.percentile(y_reg, 25)
p50 = np.percentile(y_reg, 50)

y_cls = []

for v in y_reg:
    
    if v <= p10:
        y_cls.append(0)
        
    elif v <= p25:
        y_cls.append(1)
        
    elif v <= p50:
        y_cls.append(2)
        
    else:
        y_cls.append(3)

y_cls = np.array(y_cls)

print(np.bincount(y_cls))


[ 500  750 1250 2500]



## Normalize Input


In [8]:

scl = StandardScaler()

x = scl.fit_transform(x)

joblib.dump(scl, os.path.join(sav_dir, "scl.pkl"))


['runs_cantilever\\scl.pkl']


## Train Test Split


In [9]:

xr_tr, xr_te, yr_tr, yr_te = train_test_split(
    x,
    y_reg,
    test_size=0.2,
    random_state=42
)

xc_tr, xc_te, yc_tr, yc_te = train_test_split(
    x,
    y_cls,
    test_size=0.2,
    random_state=42
)



## Dataset Class


In [10]:

class Dat(Dataset):
    
    def __init__(self, x, y):
        
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y)
        
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        
        a = self.x[idx].unsqueeze(0)
        b = self.y[idx]
        
        return a, b



## CNN Models


In [11]:

class RegNet(nn.Module):
    
    def __init__(self):
        
        super().__init__()
        
        self.net = nn.Sequential(
            
            nn.Conv1d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Flatten(),
            
            nn.Linear(32 * 56, 128),
            nn.ReLU(),
            
            nn.Linear(128, 1)
        )
        
    def forward(self, x):
        return self.net(x)


class ClsNet(nn.Module):
    
    def __init__(self):
        
        super().__init__()
        
        self.net = nn.Sequential(
            
            nn.Conv1d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Flatten(),
            
            nn.Linear(32 * 56, 128),
            nn.ReLU(),
            
            nn.Linear(128, 4)
        )
        
    def forward(self, x):
        return self.net(x)



## DataLoaders


In [12]:

bat = 32

reg_tr = DataLoader(Dat(xr_tr, yr_tr), batch_size=bat, shuffle=True)
reg_te = DataLoader(Dat(xr_te, yr_te), batch_size=bat)

cls_tr = DataLoader(Dat(xc_tr, yc_tr), batch_size=bat, shuffle=True)
cls_te = DataLoader(Dat(xc_te, yc_te), batch_size=bat)



## Train Regression Model


In [13]:

dev = "cuda" if torch.cuda.is_available() else "cpu"

reg = RegNet().to(dev)

los_fn = nn.MSELoss()

opt = torch.optim.Adam(reg.parameters(), lr=0.001)

ep = 20

for e in range(ep):
    
    reg.train()
    
    ttl = 0
    
    for xb, yb in reg_tr:
        
        xb = xb.to(dev)
        yb = yb.to(dev).float().view(-1, 1)
        
        out = reg(xb)
        
        los = los_fn(out, yb)
        
        opt.zero_grad()
        los.backward()
        opt.step()
        
        ttl += los.item()
    
    print(f"ep {e+1} loss {ttl:.4f}")


ep 1 loss 13249938603376640.0000
ep 2 loss 12534736475914240.0000
ep 3 loss 8464941161906176.0000
ep 4 loss 3027030255861760.0000
ep 5 loss 2025482153361408.0000
ep 6 loss 2017831485440000.0000
ep 7 loss 2017411295870976.0000
ep 8 loss 2015986618728448.0000
ep 9 loss 2013511666368512.0000
ep 10 loss 2013814465757184.0000
ep 11 loss 2013213597106176.0000
ep 12 loss 2012708959158272.0000
ep 13 loss 2012623278702592.0000
ep 14 loss 2011981561200640.0000
ep 15 loss 2012217937494016.0000
ep 16 loss 2011545495928832.0000
ep 17 loss 2010535260323840.0000
ep 18 loss 2010157440040960.0000
ep 19 loss 2008232496463872.0000
ep 20 loss 2008080625172480.0000



## Evaluate Regression


In [14]:

reg.eval()

prd = []
act = []

with torch.no_grad():
    
    for xb, yb in reg_te:
        
        xb = xb.to(dev)
        
        out = reg(xb).cpu().numpy().flatten()
        
        prd.extend(out)
        act.extend(yb.numpy())

mae = mean_absolute_error(act, prd)
rmse = np.sqrt(mean_squared_error(act, prd))
r2 = r2_score(act, prd)

print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)


MAE : 2569830.791
RMSE: 3895556.9911618517
R2  : -0.003714987078581311



## Train Classification Model


In [15]:

cls = ClsNet().to(dev)

los_fn2 = nn.CrossEntropyLoss()

opt2 = torch.optim.Adam(cls.parameters(), lr=0.001)

for e in range(ep):
    
    cls.train()
    
    ttl = 0
    
    for xb, yb in cls_tr:
        
        xb = xb.to(dev)
        yb = yb.to(dev).long()
        
        out = cls(xb)
        
        los = los_fn2(out, yb)
        
        opt2.zero_grad()
        los.backward()
        opt2.step()
        
        ttl += los.item()
    
    print(f"ep {e+1} loss {ttl:.4f}")


ep 1 loss 152.8438
ep 2 loss 150.0195
ep 3 loss 146.5882
ep 4 loss 140.2082
ep 5 loss 133.4280
ep 6 loss 125.5387
ep 7 loss 116.9180
ep 8 loss 107.3939
ep 9 loss 97.5794
ep 10 loss 87.0306
ep 11 loss 76.5395
ep 12 loss 62.0492
ep 13 loss 50.2962
ep 14 loss 35.6974
ep 15 loss 24.3663
ep 16 loss 14.2269
ep 17 loss 8.7994
ep 18 loss 5.1663
ep 19 loss 2.7828
ep 20 loss 1.7293



## Evaluate Classification


In [16]:

cls.eval()

prd = []
act = []

with torch.no_grad():
    
    for xb, yb in cls_te:
        
        xb = xb.to(dev)
        
        out = cls(xb)
        
        pp = torch.argmax(out, dim=1).cpu().numpy()
        
        prd.extend(pp)
        act.extend(yb.numpy())

acc = accuracy_score(act, prd)

cm = confusion_matrix(act, prd)

print("ACC :", acc)

print("\nCM:\n")
print(cm)

print("\nReport:\n")
print(classification_report(act, prd))


ACC : 0.356

CM:

[[ 13  24  16  50]
 [ 20  19  34  74]
 [ 26  39  49 138]
 [ 47  69 107 275]]

Report:

              precision    recall  f1-score   support

           0       0.12      0.13      0.12       103
           1       0.13      0.13      0.13       147
           2       0.24      0.19      0.21       252
           3       0.51      0.55      0.53       498

    accuracy                           0.36      1000
   macro avg       0.25      0.25      0.25      1000
weighted avg       0.35      0.36      0.35      1000




## Save Models


In [ ]:

torch.save(reg.state_dict(), os.path.join(sav_dir, "reg.pth"))

torch.save(cls.state_dict(), os.path.join(sav_dir, "cls.pth"))

print("saved")
